# RSNA Knee: Labels, Co-occurrence and MRI Views

A first look at the data for the [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection) competition. Before training anything it is worth knowing what the files contain, how the twelve labels behave, what mix of scanners and protocols we are dealing with, and what the images actually look like.

Only the official competition files are used. The notebook runs on CPU in a few minutes. Where reading every DICOM would take too long it reads a fixed, seeded sample, so the numbers are the same on every run.

**Contents**
1. Files and how they fit together
2. Radiology reports
3. Labels: prevalence, co-occurrence and combinations
4. The series table: planes, sequence types and slice counts
5. DICOM headers: scanners, resolution, protocols and orientation
6. Images: views, a slice strip, intensities and labelled examples
7. Test set notes and pitfalls
8. Summary

In [ ]:
import os
import re
import time
import warnings
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import pydicom
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 160)
T0 = time.time()
SEED = 42

# Kaggle mounts the competition under one of the first two paths. RSNA_KNEE_DIR lets the
# same notebook run against a local copy of the data.
CANDIDATES = ['/kaggle/input/competitions/rsna-knee-abnormality-detection',
              '/kaggle/input/rsna-knee-abnormality-detection',
              os.environ.get('RSNA_KNEE_DIR', '')]
DATA_DIR = next((p for p in CANDIDATES if p and os.path.isfile(os.path.join(p, 'train.csv'))), None)
if DATA_DIR is None:
    raise FileNotFoundError('Competition data not found: add the competition as an input or set RSNA_KNEE_DIR')
print('data:', DATA_DIR)

# How much DICOM to read. These keep the whole notebook to a few minutes on a CPU session.
N_HEADER_STUDIES = 800    # studies whose series headers are read (one file per series)
N_ORDER_SERIES = 40       # series whose slices are all read to compare file order with anatomy
N_INTENSITY_SERIES = 10   # mid slices per (plane, sequence type) group for the intensity plots

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
PLANES = ['Sagittal', 'Coronal', 'Axial']

# One fixed colour per plane and per sequence type, used the same way in every plot
PLANE_COLORS = {'Sagittal': '#2a78d6', 'Coronal': '#eb6834', 'Axial': '#1baf7a'}
FS_COLORS = {1: '#4a3aa7', 0: '#eda100'}
FS_NAMES = {1: 'fluid-sensitive', 0: 'not fluid-sensitive'}
BAR = '#2a78d6'
POS_COLOR, NEG_COLOR = '#e34948', '#898781'
SEQ_CMAP = mcolors.LinearSegmentedColormap.from_list('seq', ['#f7fafd', '#b7d3f6', '#5598e7', '#1c5cab', '#0d366b'])
DIV_CMAP = mcolors.LinearSegmentedColormap.from_list('div', ['#1c5cab', '#86b6ef', '#f0efec', '#ef9a8f', '#b8302f'])

sns.set_theme(style='whitegrid', context='notebook', font_scale=0.95)
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 90, 'axes.titlesize': 11,
                     'axes.titleweight': 'bold', 'axes.edgecolor': '#c3c2b7',
                     'grid.color': '#e1e0d9', 'axes.labelcolor': '#52514e',
                     'xtick.color': '#52514e', 'ytick.color': '#52514e'})

FIG_DIR = 'figures'  # a PNG copy of every figure goes here (the Output tab on Kaggle)
os.makedirs(FIG_DIR, exist_ok=True)


def finish(fig, name):
    """Lay out, save a PNG copy and show a figure."""
    fig.tight_layout()
    try:
        fig.savefig(os.path.join(FIG_DIR, name + '.png'), bbox_inches='tight')
    except Exception as e:
        print('could not save figure', name, e)
    plt.show()
    plt.close(fig)


def run(title, fn):
    """Run one analysis block. If an odd file breaks it, say so and carry on with the rest."""
    t = time.time()
    try:
        out = fn()
    except Exception as e:
        plt.close('all')
        print(f'[{title}] skipped: {type(e).__name__}: {e}')
        return None
    if time.time() - t > 10:
        print(f'[{title}] took {time.time() - t:.0f}s')
    return out


def series_dir(split, study, series):
    return os.path.join(DATA_DIR, f'{split}_series', str(study), str(series))


def list_dcm(path):
    try:
        return sorted(f for f in os.listdir(path) if f.lower().endswith('.dcm'))
    except OSError:
        return []


def fnum(x):
    """float, or NaN for a DICOM value that is missing, empty or malformed."""
    try:
        v = float(x)
        return v if np.isfinite(v) else np.nan
    except Exception:
        return np.nan


def two(v):
    """(row, col) from a DICOM pair such as PixelSpacing; a single value is used for both."""
    if v is None:
        return np.nan, np.nan
    try:
        vals = [fnum(x) for x in v] if isinstance(v, (list, tuple, pydicom.multival.MultiValue)) else [fnum(v)]
    except Exception:
        return np.nan, np.nan
    vals = (vals + vals)[:2] if vals else [np.nan, np.nan]
    return vals[0], vals[1]


def plane_from_iop(iop):
    """Plane implied by ImageOrientationPatient and the tilt (degrees) of the slice normal from that axis."""
    try:
        o = np.asarray([float(v) for v in iop], dtype=float)
        n = np.cross(o[:3], o[3:])
        n = n / np.linalg.norm(n)
        k = int(np.argmax(np.abs(n)))
        return PLANES[k], float(np.degrees(np.arccos(min(1.0, abs(n[k])))))
    except Exception:
        return 'Unknown', np.nan


def human_size(n):
    for unit in ['B', 'kB', 'MB', 'GB']:
        if n < 1000:
            return f'{n:.0f} {unit}' if unit == 'B' else f'{n:.1f} {unit}'
        n /= 1000
    return f'{n:.1f} TB'

## 1. Files and how they fit together

The competition folder holds a few CSV files plus two DICOM folders. The table lists each top-level entry with its size (for folders, the number of study folders inside).

In [ ]:
rows = []
for name in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, name)
    if os.path.isdir(p):
        rows.append({'entry': name + '/', 'kind': 'folder', 'size or contents': f'{len(os.listdir(p)):,} study folders'})
    else:
        rows.append({'entry': name, 'kind': 'file', 'size or contents': human_size(os.path.getsize(p))})
display(pd.DataFrame(rows).set_index('entry'))


def read_csv_opt(name):
    p = os.path.join(DATA_DIR, name)
    if not os.path.isfile(p):
        print('missing:', name)
        return None
    return pd.read_csv(p, dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})


train = read_csv_opt('train.csv')
train_series = read_csv_opt('train_series.csv')
test = read_csv_opt('test.csv')
test_series = read_csv_opt('test_series.csv')
sample_sub = read_csv_opt('sample_submission.csv')

overview = []
for name, df in [('train.csv', train), ('train_series.csv', train_series), ('test.csv', test),
                 ('test_series.csv', test_series), ('sample_submission.csv', sample_sub)]:
    if df is None:
        continue
    overview.append({'file': name, 'rows': len(df), 'columns': df.shape[1],
                     'unique studies': df['StudyInstanceUID'].nunique() if 'StudyInstanceUID' in df else '-',
                     'unique series': df['SeriesInstanceUID'].nunique() if 'SeriesInstanceUID' in df else '-'})
display(pd.DataFrame(overview).set_index('file'))
print('train.csv columns:       ', list(train.columns))
print('train_series.csv columns:', list(train_series.columns))

The data is organised as **study → series → slice**:

- a **study** is one knee examination (one row in `train.csv`, one folder under `train_series/`),
- a **series** is one MRI acquisition inside it, such as a sagittal fat-suppressed sequence (one row in `train_series.csv`, one sub-folder),
- a **slice** is one 2D image, stored as one `.dcm` file: `train_series/<StudyInstanceUID>/<SeriesInstanceUID>/<SOPInstanceUID>.dcm`.

The cell below checks that the CSVs and the folders agree and counts the slice files in every series folder. Listing folders is cheap compared with reading files, so this part covers all series rather than a sample. A small thread pool helps because most of the time goes into waiting on the file system.

In [ ]:
t = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    train_series['n_slices'] = list(pool.map(
        lambda ids: len(list_dcm(series_dir('train', *ids))),
        zip(train_series.StudyInstanceUID, train_series.SeriesInstanceUID)))
study_root = os.path.join(DATA_DIR, 'train_series')
study_dirs = set(os.listdir(study_root)) if os.path.isdir(study_root) else set()
csv_series = train_series.groupby('StudyInstanceUID').SeriesInstanceUID.apply(set)


def extra_series_dirs(st):
    try:
        return len(set(os.listdir(os.path.join(study_root, st))) - csv_series[st])
    except OSError:
        return 0


with ThreadPoolExecutor(max_workers=16) as pool:
    extra_dirs = sum(pool.map(extra_series_dirs, sorted(study_dirs & set(csv_series.index))))
print(f'listed {len(train_series):,} series folders in {time.time() - t:.0f}s')

tr_ids, ser_ids = set(train.StudyInstanceUID), set(train_series.StudyInstanceUID)
checks = {
    'studies in train.csv': len(tr_ids),
    'studies in train_series.csv': len(ser_ids),
    'train.csv studies with no series row': len(tr_ids - ser_ids),
    'series rows whose study is not in train.csv': len(ser_ids - tr_ids),
    'study folders on disk': len(study_dirs & tr_ids),
    'series folders with at least one .dcm': int((train_series.n_slices > 0).sum()),
    'series sub-folders not listed in the CSV': extra_dirs,
    'slice files (.dcm) in total': int(train_series.n_slices.sum()),
}
display(pd.Series(checks).map('{:,}'.format).to_frame('count'))
ON_DISK = train_series.n_slices > 0

## 2. Radiology reports

Every training study comes with the free-text report written by the radiologist. The data page says the reports are in several languages and are **not** provided for the test studies. Here is the start of `train.csv` (reports cut to 90 characters), followed by report length and a rough guess at the language.

The language guess is a simple stop-word count plus a check for Cyrillic and Greek script, so treat it as approximate.

In [ ]:
preview = train.head(4).copy()
preview['Report'] = preview['Report'].astype(str).str.replace(r'\s+', ' ', regex=True).str.slice(0, 90) + ' ...'
display(preview)

STOPWORDS = {
    'English': 'the of and with is are there no without in to'.split(),
    'Spanish': 'de la el con sin los del las por y se'.split(),
    'Portuguese': 'do da com sem não os ao na das dos em'.split(),
    'German': 'der die und mit ist kein keine im des das nicht'.split(),
    'French': 'le les des avec sans et du une est pas'.split(),
    'Italian': 'di il della con senza del non sono delle nella'.split(),
    'Dutch': 'van het en met geen een niet op'.split(),
    'Turkish': 've ile bir yok olarak izlendi mevcut normaldir'.split(),
    'Croatian / Serbian': 'je i se u na bez nije od su uz sa te'.split(),
}


def guess_language(text):
    s = str(text)
    letters = [c for c in s if c.isalpha()]
    if not letters:
        return 'Unknown'
    if sum('Ѐ' <= c <= 'ӿ' for c in letters) / len(letters) > 0.3:
        return 'Cyrillic script'
    if sum('Ͱ' <= c <= 'Ͽ' for c in letters) / len(letters) > 0.3:
        return 'Greek'
    words = re.findall(r'[^\W\d_]+', s.lower())
    cnt = Counter(words)
    scores = {k: sum(cnt[w] for w in v) / max(len(words), 1) for k, v in STOPWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] >= 0.03 else 'Other'


train['report_chars'] = train['Report'].astype(str).str.len()
train['language'] = train['Report'].map(guess_language)
has_all = train[LABELS].notna().all(axis=1)


def plot_reports():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={'width_ratios': [1.2, 1]})
    ax = axes[0]
    ax.hist(train['report_chars'], bins=60, color=BAR, edgecolor='white', linewidth=0.5)
    med = train['report_chars'].median()
    ax.axvline(med, color='#0b0b0b', lw=1, ls='--')
    ax.text(med, ax.get_ylim()[1] * 0.92, f'  median {med:,.0f}', color='#0b0b0b', fontsize=9)
    ax.set(title='Report length', xlabel='characters', ylabel='studies')
    vc = train['language'].value_counts()
    ax = axes[1]
    ax.barh(vc.index[::-1], vc.values[::-1], color=BAR, height=0.7)
    for y, v in enumerate(vc.values[::-1]):
        ax.text(v, y, f' {v:,}', va='center', fontsize=9, color='#52514e')
    ax.set(title='Report language (rough guess)', xlabel='studies')
    ax.set_xlim(0, vc.max() * 1.15)
    finish(fig, '02_reports')
    tab = pd.DataFrame({'all studies': vc, 'labelled studies': train.loc[has_all, 'language'].value_counts()})
    display(tab.fillna(0).astype(int))


run('reports', plot_reports)

tokens = Counter(t for r in train['Report'].astype(str) for t in re.findall(r'\[[A-Z][A-Z _]{1,30}\]', r))
n_with = train['Report'].astype(str).str.contains(r'\[[A-Z][A-Z _]{1,30}\]', regex=True).sum()
print(f'{n_with:,} reports contain a bracketed placeholder; most common:', tokens.most_common(6))

**Observations**

- Reports are fairly short (median just under 1,000 characters) and come in many languages. English is the largest group at about 40%, followed by Spanish, Turkish, Croatian or Serbian, Greek and German. The guess is rough, but the spread is clearly wide.
- The 58 labelled studies (next section) have a similar language mix, with English a bit more common (28 of 58).
- About one report in seven contains de-identification placeholders such as `[DATE]`, `[TIME]` or `[REDACTED]`.
- Test studies come without a report, so at scoring time only the images and the series table are available.

## 3. Labels

`train.csv` has twelve binary label columns, but only a small subset of studies has them filled in. The data page says so directly: the rest of the training studies come with the report only. Everything in this section is computed on the labelled studies.

In [ ]:
has_any = train[LABELS].notna().any(axis=1)
print(f'studies with all 12 labels:    {has_all.sum():,}')
print(f'studies with some, not all:    {(has_any & ~has_all).sum():,}')
print(f'studies with no labels at all: {(~has_any).sum():,}')
lab = train.loc[has_all, ['StudyInstanceUID'] + LABELS].copy()
lab[LABELS] = lab[LABELS].astype(int)
Y = lab[LABELS]
N_LAB = len(Y)


def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def plot_prevalence():
    k = Y.sum().sort_values()
    p = k / N_LAB
    lo, hi = zip(*[wilson(v, N_LAB) for v in k])
    fig, ax = plt.subplots(figsize=(10, 5.2))
    ax.barh(k.index, p.values, color=BAR, height=0.65)
    ax.errorbar(p.values, np.arange(len(k)), xerr=[p.values - np.array(lo), np.array(hi) - p.values],
                fmt='none', ecolor='#52514e', elinewidth=1, capsize=3)
    for y, (v, pp, h) in enumerate(zip(k.values, p.values, hi)):
        ax.text(h + 0.01, y, f'{v}/{N_LAB} ({pp:.0%})', va='center', fontsize=9, color='#52514e')
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.set(title=f'Share of labelled studies positive for each finding (n = {N_LAB}, bars show 95% Wilson intervals)',
           xlabel='positive share')
    finish(fig, '03_label_prevalence')


run('label prevalence', plot_prevalence)

The whiskers are 95% confidence intervals. With this few labelled studies they are wide, so a difference of a few percentage points between two findings is not meaningful. The data page also warns that prevalence is not guaranteed to match between the training data and the public and private test sets.

Next: how many findings a study has, how often findings appear together (counts) and how strongly they are correlated (phi, which is just the Pearson correlation of two 0/1 columns).

In [ ]:
def plot_cooccurrence():
    card = Y.sum(axis=1).value_counts().reindex(range(0, 13), fill_value=0)
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.bar(card.index, card.values, color=BAR, width=0.75)
    for x, v in zip(card.index, card.values):
        if v:
            ax.text(x, v, str(v), ha='center', va='bottom', fontsize=9, color='#52514e')
    ax.set_xticks(range(0, 13))
    ax.set(title='Number of positive findings per labelled study', xlabel='positive findings', ylabel='studies')
    finish(fig, '03_label_cardinality')
    print(f'mean {Y.sum(axis=1).mean():.2f} positive findings per study; '
          f'{(Y.sum(axis=1) == 0).sum()} studies with none')

    co = Y.T.dot(Y)
    phi = Y.corr()
    fig, axes = plt.subplots(1, 2, figsize=(17, 7))
    sns.heatmap(co, annot=True, fmt='d', cmap=SEQ_CMAP, cbar_kws={'shrink': 0.7, 'label': 'studies'},
                linewidths=1, linecolor='white', square=True, ax=axes[0], annot_kws={'fontsize': 8})
    axes[0].set_title('Co-occurrence counts (diagonal = studies with that finding)')
    mask = np.eye(len(LABELS), dtype=bool)
    sns.heatmap(phi, annot=True, fmt='.2f', cmap=DIV_CMAP, vmin=-0.6, vmax=0.6, center=0, mask=mask,
                cbar_kws={'shrink': 0.7, 'label': 'phi'}, linewidths=1, linecolor='white', square=True,
                ax=axes[1], annot_kws={'fontsize': 8})
    axes[1].set_title('Phi correlation between findings (diagonal hidden)')
    for ax in axes:
        ax.grid(False)
        ax.tick_params(axis='x', rotation=45)
        plt.setp(ax.get_xticklabels(), ha='right')
    finish(fig, '03_label_cooccurrence')

    pairs = [(LABELS[i], LABELS[j], phi.iloc[i, j], int(co.iloc[i, j]))
             for i in range(len(LABELS)) for j in range(i + 1, len(LABELS))]
    pairs = pd.DataFrame(pairs, columns=['finding A', 'finding B', 'phi', 'both positive'])
    print('Strongest positive and negative pairs:')
    display(pd.concat([pairs.nlargest(6, 'phi'), pairs.nsmallest(3, 'phi')]).round(2).reset_index(drop=True))


run('co-occurrence', plot_cooccurrence)

In [ ]:
combo = Y.apply(lambda r: ' + '.join(l for l in LABELS if r[l] == 1) or '(no findings)', axis=1)
vc = combo.value_counts()
print(f'{vc.size} distinct combinations among {N_LAB} labelled studies; {int((vc == 1).sum())} occur only once')
display(vc.head(12).rename_axis('positive findings').to_frame('studies'))

**Observations**

- Effusion is the most common finding (35 of 58, 60%) and MCL the rarest (9 of 58, 16%). With 58 studies the 95% intervals are around 20 to 25 percentage points wide, so the ranking in the middle of the chart is loose.
- None of the 58 labelled studies is completely normal: every one has at least one positive finding, and the average is about four. The positive rates here may well be higher than in the rest of the data.
- The correlations fall roughly into two groups. Degenerative findings go together: medial OA with Baker's cyst (phi 0.48) and with both meniscus tears (0.42 and 0.41), lateral OA with lateral meniscus tear (0.42). Acute injury findings go together too: ACL with contusion (0.38) and fracture (0.27), contusion with fracture (0.33). Between the two groups the correlations are mostly negative, for example medial OA and contusion (-0.33).
- Effusion and synovitis appear together in 22 studies (phi 0.40). MCL never appears together with medial OA or a Baker's cyst in this set.
- There are 54 different combinations among 58 studies and 50 of them occur only once, so there is no small set of typical label patterns.

## 4. The series table: planes, sequence types and slice counts

`train_series.csv` describes every series with three columns: the anatomical plane, whether the sequence is fluid-sensitive (T2, PD, STIR and similar, where fluid is bright) and whether it uses fat suppression. Together with the slice counts from section 1 this is all we know about a series without opening a DICOM.

In [ ]:
S = train_series


def plot_series_table():
    per_study = S.groupby('StudyInstanceUID').size()
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.3), gridspec_kw={'width_ratios': [1.1, 1, 0.9]})
    vc = per_study.value_counts().sort_index()
    axes[0].bar(vc.index, vc.values, color=BAR, width=0.75)
    axes[0].set_xticks(vc.index)
    for x, v in zip(vc.index, vc.values):
        axes[0].text(x, v, f'{v:,}', ha='center', va='bottom', fontsize=8, color='#52514e')
    axes[0].set_ylim(0, vc.max() * 1.1)
    axes[0].set(title='Series per study', xlabel='series in the study', ylabel='studies')

    ct = pd.crosstab(S.Anatomical_Plane, S.Fluid_Sensitive).reindex(PLANES).fillna(0)
    x = np.arange(len(PLANES))
    w = 0.38
    for j, fs in enumerate([1, 0]):
        vals = ct[fs].values if fs in ct else np.zeros(len(PLANES))
        axes[1].bar(x + (j - 0.5) * w, vals, w * 0.92, color=FS_COLORS[fs], label=FS_NAMES[fs])
        for xi, v in zip(x, vals):
            axes[1].text(xi + (j - 0.5) * w, v, f'{int(v):,}', ha='center', va='bottom', fontsize=8, color='#52514e')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(PLANES)
    axes[1].set(title='Series by plane and sequence type', ylabel='series')
    axes[1].set_ylim(0, ct.values.max() * 1.25)
    axes[1].legend(frameon=False, loc='upper center', ncol=2)

    have = S.groupby(['StudyInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive']).size().unstack([1, 2]).notna()
    cov = pd.DataFrame({p: [have[(p, fs)].mean() if (p, fs) in have else 0.0 for fs in [1, 0]] for p in PLANES},
                       index=[FS_NAMES[1], FS_NAMES[0]])
    sns.heatmap(cov, annot=True, fmt='.0%', cmap=SEQ_CMAP, vmin=0, vmax=1, cbar=False, linewidths=2,
                linecolor='white', ax=axes[2], annot_kws={'fontsize': 11})
    axes[2].set_title('Studies with at least one series of each kind')
    axes[2].set_ylabel('')
    finish(fig, '04_series_table')

    planes_per_study = S.groupby('StudyInstanceUID').Anatomical_Plane.apply(set)
    print('studies missing a plane entirely:',
          {p: int((~planes_per_study.map(lambda s, p=p: p in s)).sum()) for p in PLANES})
    print('studies with all three planes:', int(planes_per_study.map(lambda s: set(PLANES) <= s).sum()),
          'of', len(planes_per_study))
    print('\nFluid_Sensitive vs Fat_Suppression (series):')
    display(pd.crosstab(S.Fluid_Sensitive, S.Fat_Suppression))
    print('identical in', int((S.Fluid_Sensitive == S.Fat_Suppression).sum()), 'of', len(S), 'rows')


run('series table', plot_series_table)

In [ ]:
def plot_slices():
    s = S[ON_DISK]
    fig, ax = plt.subplots(figsize=(13, 4.2))
    bins = np.arange(0, 102, 2)
    for p in PLANES:
        v = s.loc[s.Anatomical_Plane == p, 'n_slices'].clip(upper=100)
        ax.hist(v, bins=bins, histtype='step', lw=2, color=PLANE_COLORS[p], label=f'{p} (n = {len(v):,})')
    ax.set(title='Slices per series (series with more than 100 slices are piled into the last bin)',
           xlabel='slices', ylabel='series')
    ax.set_xlim(5, 102)
    ax.legend(frameon=False)
    finish(fig, '04_slices_per_series')
    tab = s.groupby(['Anatomical_Plane', 'Fluid_Sensitive']).n_slices.describe(percentiles=[0.05, 0.5, 0.95])
    tab = tab[['count', 'min', '5%', '50%', '95%', 'max']].astype(int)
    display(tab.reindex(PLANES, level=0))
    print(f'series with more than 100 slices: {(s.n_slices > 100).sum():,}; '
          f'with fewer than 15: {(s.n_slices < 15).sum():,}')


run('slices per series', plot_slices)

**Observations**

- Most studies have five series (2,299 of 4,407). The full range is 3 to 14.
- Every study has at least one series in each of the three planes. A fluid-sensitive series is there in 94% of studies for the sagittal plane, 96% for coronal and 100% for axial. Non-fluid-sensitive axial series are uncommon (19% of studies), while a non-fluid-sensitive sagittal series is almost always present (97%).
- `Fluid_Sensitive` and `Fat_Suppression` are identical on all 24,371 training rows. The data page says they are not necessarily equivalent for every case, so the test CSV may not follow this.
- There are 819,078 slice files in total. A typical series has about 30 slices, and for most plane and sequence groups 90% of series fall between roughly 16 and 60 slices. The tail is long, though: 709 series have more than 100 slices, the longest has 320, and the 95th percentile for axial fluid-sensitive series is 144. Section 5 shows that these long series are 3D acquisitions.

## 5. DICOM headers: scanners, resolution, protocols and orientation

The DICOMs keep an allow-list of metadata tags, which says a lot about how varied the data is. Reading one header per series for every study would take a while, so this section reads the middle file of every series for a seeded random sample of studies.

In [ ]:
studies_on_disk = sorted(set(S.loc[ON_DISK, 'StudyInstanceUID']))
rng = np.random.default_rng(SEED)
sample_studies = sorted(rng.choice(studies_on_disk, size=min(N_HEADER_STUDIES, len(studies_on_disk)), replace=False))


def header_record(path):
    d = pydicom.dcmread(path, stop_before_pixels=True)
    ps_r, ps_c = two(d.get('PixelSpacing', None))
    plane, tilt = plane_from_iop(d.get('ImageOrientationPatient', None))
    try:
        ts = d.file_meta.TransferSyntaxUID.name
    except Exception:
        ts = 'unknown'
    return {
        'transfer_syntax': ts,
        'manufacturer': str(d.get('Manufacturer', '') or ''),
        'model': str(d.get('ManufacturerModelName', '') or ''),
        'field_T': fnum(d.get('MagneticFieldStrength', None)),
        'rows': fnum(d.get('Rows', None)), 'cols': fnum(d.get('Columns', None)),
        'ps_row': ps_r, 'ps_col': ps_c,
        'thickness': fnum(d.get('SliceThickness', None)),
        'spacing': fnum(d.get('SpacingBetweenSlices', None)),
        'TE': fnum(d.get('EchoTime', None)), 'TR': fnum(d.get('RepetitionTime', None)),
        'acq': str(d.get('MRAcquisitionType', '') or ''),
        'description': str(d.get('SeriesDescription', '') or ''),
        'sex': str(d.get('PatientSex', '') or ''),
        'patient_id': str(d.get('PatientID', '') or ''),
        'laterality': str(d.get('Laterality', '') or ''),
        'contrast_agent': str(d.get('ContrastBolusAgent', '') or ''),
        'photometric': str(d.get('PhotometricInterpretation', '') or ''),
        'bits_stored': fnum(d.get('BitsStored', None)),
        'frames': fnum(d.get('NumberOfFrames', None)),
        'iop_plane': plane, 'tilt_deg': tilt,
    }


def series_header(r):
    """Header of the middle file of one series, plus its CSV columns; an exception name if it fails."""
    sdir = series_dir('train', r.StudyInstanceUID, r.SeriesInstanceUID)
    files = list_dcm(sdir)
    try:
        rec = header_record(os.path.join(sdir, files[len(files) // 2]))
    except Exception as e:
        return type(e).__name__
    rec.update(StudyInstanceUID=r.StudyInstanceUID, SeriesInstanceUID=r.SeriesInstanceUID,
               plane=r.Anatomical_Plane, fs=r.Fluid_Sensitive, n_slices=r.n_slices)
    return rec


t = time.time()
sub = S[S.StudyInstanceUID.isin(set(sample_studies)) & ON_DISK]
with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(series_header, sub.itertuples(index=False)))
recs = [r for r in results if isinstance(r, dict)]
fails = Counter(r for r in results if isinstance(r, str))
HDR_COLS = ['transfer_syntax', 'manufacturer', 'model', 'field_T', 'rows', 'cols', 'ps_row', 'ps_col', 'thickness',
            'spacing', 'TE', 'TR', 'acq', 'description', 'sex', 'patient_id', 'laterality', 'contrast_agent',
            'photometric', 'bits_stored', 'frames', 'iop_plane', 'tilt_deg', 'StudyInstanceUID',
            'SeriesInstanceUID', 'plane', 'fs', 'n_slices']
hdr = pd.DataFrame(recs, columns=HDR_COLS)
print(f'read {len(hdr):,} series headers from {len(sample_studies)} studies in {time.time() - t:.0f}s; '
      f'failures: {dict(fails) or "none"}')


def vendor(m):
    m = m.lower().strip()
    if 'siemens' in m:
        return 'Siemens'
    if m.startswith('ge') or 'general electric' in m:
        return 'GE'
    if 'philips' in m:
        return 'Philips'
    if 'toshiba' in m or 'canon' in m:
        return 'Canon / Toshiba'
    if 'hitachi' in m or 'fujifilm' in m:
        return 'Hitachi / Fujifilm'
    return 'Missing' if m in ('', 'none', 'nan') else 'Other'


def field_group(f):
    if not np.isfinite(f):
        return 'Missing'
    if f > 100:          # some scanners write Gauss instead of Tesla
        f = f / 10000
    return f'{f:g} T' if f in (0.2, 0.3, 0.35, 0.5, 1.0, 1.2, 1.5, 3.0) else ('< 1 T' if f < 1 else f'{round(f, 1):g} T')


hdr['vendor'] = hdr['manufacturer'].map(vendor)
hdr['field'] = hdr['field_T'].map(field_group)
per_study_hdr = hdr.sort_values('SeriesInstanceUID').groupby('StudyInstanceUID').first()

In [ ]:
def barh(ax, vc, title, xlabel):
    vc = vc.sort_values()
    ax.barh([str(i) for i in vc.index], vc.values, color=BAR, height=0.7)
    for y, v in enumerate(vc.values):
        ax.text(v, y, f' {v:,} ({v / vc.sum():.0%})', va='center', fontsize=9, color='#52514e')
    ax.set_xlim(0, vc.max() * 1.35)
    ax.set(title=title, xlabel=xlabel)


def plot_scanners():
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    barh(axes[0], per_study_hdr['vendor'].value_counts(), 'Scanner vendor', 'studies in the sample')
    barh(axes[1], per_study_hdr['field'].value_counts(), 'Field strength', 'studies in the sample')
    barh(axes[2], hdr['transfer_syntax'].value_counts(), 'Transfer syntax (pixel encoding)', 'series in the sample')
    finish(fig, '05_scanners')
    print('distinct scanner models in the sample:', hdr['model'].replace('', np.nan).nunique())
    display(per_study_hdr.groupby(['vendor', 'model']).size().sort_values(ascending=False).head(10).to_frame('studies'))


run('scanners', plot_scanners)

In [ ]:
def plot_geometry():
    h = hdr.copy()
    h['matrix'] = h['rows'].fillna(0).astype(int).astype(str) + ' x ' + h['cols'].fillna(0).astype(int).astype(str)
    h['fov_mm'] = h['rows'] * h['ps_row']
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.3))
    barh(axes[0], h['matrix'].value_counts().head(10), 'Most common image sizes (rows x columns)', 'series')
    for p in PLANES:
        v = h.loc[h.plane == p, 'ps_row'].dropna()
        axes[1].hist(v.clip(upper=1.0), bins=np.linspace(0.1, 1.0, 46), histtype='step', lw=2,
                     color=PLANE_COLORS[p], label=p)
        v = h.loc[h.plane == p, 'fov_mm'].dropna()
        axes[2].hist(v.clip(upper=300), bins=np.linspace(80, 300, 45), histtype='step', lw=2,
                     color=PLANE_COLORS[p], label=p)
    axes[1].set(title='In-plane pixel spacing (values above 1 mm in last bin)', xlabel='mm per pixel', ylabel='series')
    axes[2].set(title='Field of view along the rows (rows x spacing)', xlabel='mm', ylabel='series')
    for ax in axes[1:]:
        ax.legend(frameon=False)
    finish(fig, '05_geometry')
    print(f"{h['matrix'].nunique()} distinct image sizes; pixel spacing median {h['ps_row'].median():.2f} mm "
          f"(5th to 95th percentile {h['ps_row'].quantile(0.05):.2f} to {h['ps_row'].quantile(0.95):.2f} mm); "
          f"non-square pixels in {(np.abs(h['ps_row'] - h['ps_col']) > 1e-3).sum()} series; "
          f"non-square images (rows != columns) in {(h['rows'] != h['cols']).sum()} series")


run('geometry', plot_geometry)

In [ ]:
def plot_thickness():
    h = hdr.copy()
    h['gap'] = h['spacing'] - h['thickness']
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    for p in PLANES:
        v = h.loc[h.plane == p, 'thickness'].dropna()
        axes[0].hist(v.clip(upper=6), bins=np.arange(0, 6.25, 0.25), histtype='step', lw=2, color=PLANE_COLORS[p], label=p)
        v = h.loc[h.plane == p, 'gap'].dropna()
        axes[1].hist(v.clip(-1.5, 2.5), bins=np.arange(-1.5, 2.6, 0.1), histtype='step', lw=2, color=PLANE_COLORS[p], label=p)
    axes[0].set(title='Slice thickness (values above 6 mm in last bin)', xlabel='mm', ylabel='series')
    axes[1].set(title='Spacing between slices minus thickness (negative = overlap)', xlabel='mm', ylabel='series')
    for ax in axes:
        ax.legend(frameon=False)
    finish(fig, '05_thickness')
    print('slice thickness by plane (mm):')
    display(h.groupby('plane').thickness.describe(percentiles=[0.05, 0.5, 0.95])[['count', '5%', '50%', '95%']]
            .reindex(PLANES).round(2))
    print(f"SpacingBetweenSlices missing in {h['spacing'].isna().mean():.0%} of series")


run('thickness', plot_thickness)

Echo time (TE) and repetition time (TR) are the two numbers that decide an MRI sequence's contrast. Plotting them against the `Fluid_Sensitive` flag from the CSV shows how the flag lines up with the actual acquisition parameters.

In [ ]:
def plot_te_tr():
    h = hdr.dropna(subset=['TE', 'TR'])
    h = h[(h.TE > 0) & (h.TR > 0)]
    fig, ax = plt.subplots(figsize=(10, 6))
    for fs in [0, 1]:
        v = h[h.fs == fs]
        ax.scatter(v.TR, v.TE, s=12, alpha=0.5, color=FS_COLORS[fs], edgecolor='white', linewidth=0.3,
                   label=f'{FS_NAMES[fs]} (n = {len(v):,})')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(3, 20000)
    ax.set_ylim(1, 200)
    plain = plt.FuncFormatter(lambda v, _: f'{v:g}')
    ax.xaxis.set_major_formatter(plain)
    ax.yaxis.set_major_formatter(plain)
    ax.set(title='Echo time vs repetition time, coloured by the Fluid_Sensitive flag',
           xlabel='TR (ms, log scale)', ylabel='TE (ms, log scale)')
    ax.legend(frameon=False, markerscale=2, loc='lower right')
    finish(fig, '05_te_tr')
    outside = int(((h.TE < 1) | (h.TE > 200) | (h.TR < 3) | (h.TR > 20000)).sum())
    print(f'{len(h):,} series with TE and TR; {outside} fall outside the plotted range')
    print('median TE / TR (ms) by flag:')
    display(h.groupby('fs')[['TE', 'TR']].median().rename(index=FS_NAMES).round(1))
    f0 = h[h.fs == 0]
    print(f'flag = 0 series with long TR (> 1500 ms, a PD or T2 type of contrast): '
          f'{(f0.TR > 1500).sum():,} of {len(f0):,} ({(f0.TR > 1500).mean():.0%})')


run('TE/TR', plot_te_tr)

`Anatomical_Plane` in the CSV can be compared with the plane implied by the slice orientation stored in each header (`ImageOrientationPatient`). The tilt is the angle between the slice normal and the nearest scanner axis, which shows how oblique the acquisitions are.

In [ ]:
def plot_orientation():
    ct = pd.crosstab(hdr['plane'], hdr['iop_plane']).reindex(index=PLANES)
    print('rows: Anatomical_Plane from the CSV, columns: plane from ImageOrientationPatient')
    display(ct)
    agree = (hdr['plane'] == hdr['iop_plane']).mean()
    print(f'agreement: {agree:.1%} of sampled series')
    fig, ax = plt.subplots(figsize=(11, 3.8))
    for p in PLANES:
        v = hdr.loc[hdr.plane == p, 'tilt_deg'].dropna()
        ax.hist(v.clip(upper=45), bins=np.arange(0, 46, 1), histtype='step', lw=2, color=PLANE_COLORS[p],
                label=f'{p} (median {v.median():.1f} deg)')
    ax.set(title='Tilt of the slice normal from the nearest scanner axis', xlabel='degrees', ylabel='series')
    ax.legend(frameon=False)
    finish(fig, '05_tilt')


run('orientation', plot_orientation)

In [ ]:
def plot_acq():
    h = hdr.copy()
    h['acq'] = h['acq'].replace('', 'Missing')
    print('MRAcquisitionType:', h['acq'].value_counts().to_dict())
    fig, ax = plt.subplots(figsize=(10, 3.6))
    order = [a for a in ['2D', '3D', 'Missing'] if a in set(h['acq'])]
    sns.boxplot(data=h, x='n_slices', y='acq', order=order, color='#b7d3f6', fliersize=2, ax=ax, width=0.5)
    ax.set_xscale('log')
    ax.set(title='Slices per series by acquisition type', xlabel='slices (log scale)', ylabel='')
    finish(fig, '05_acq_slices')
    display(h.groupby('acq').n_slices.describe(percentiles=[0.5, 0.95])[['count', '50%', '95%', 'max']].astype(int))


run('acquisition type', plot_acq)

In [ ]:
def show_descriptions():
    d = hdr['description'].str.strip()
    blank = (d == '').mean()
    dummy = d.str.contains('dummy', case=False).mean()
    print(f'{d.nunique():,} distinct SeriesDescription values in {len(d):,} sampled series; '
          f'blank: {blank:.1%}; placeholder "dummy" text: {dummy:.1%}')
    top = d.replace('', '(blank)').value_counts().head(20).to_frame('series')
    top['planes in CSV'] = [', '.join(sorted(set(hdr.loc[d.replace('', '(blank)') == k, 'plane']))) for k in top.index]
    display(top)


run('series descriptions', show_descriptions)

In [ ]:
def show_patient_tags():
    st = per_study_hdr
    lat = st['laterality'].str.upper().str.strip().replace({'RIGHT': 'R', 'LEFT': 'L', '': 'Missing'})
    sex = st['sex'].str.upper().str.strip().replace({'': 'Missing'})
    print('Laterality (per study):', lat.value_counts().to_dict())
    print('PatientSex (per study): ', sex.value_counts().to_dict())
    pid = hdr.groupby('patient_id').StudyInstanceUID.nunique()
    print(f'PatientID: {len(pid):,} distinct values in {hdr.StudyInstanceUID.nunique():,} sampled studies; '
          f'{int((pid > 1).sum())} IDs appear in more than one study')
    ca = st['contrast_agent'].str.strip()
    print(f'ContrastBolusAgent filled in for {int((ca != "").sum())} sampled studies')
    print('PhotometricInterpretation:', hdr['photometric'].value_counts().to_dict())
    print('BitsStored:', hdr['bits_stored'].value_counts().to_dict())
    print('NumberOfFrames > 1:', int((hdr['frames'] > 1).sum()), 'series')


run('patient tags', show_patient_tags)

**Observations** (from the 800-study header sample)

- Four vendors cover nearly everything: Siemens (44% of studies), Philips (28%), GE (22%) and Canon or Toshiba (6%), plus a handful of Hitachi or Fujifilm scans. There are 40 distinct scanner models. Field strength is 1.5 T in 59% of studies and 3 T in 35%, and missing in 6%.
- Every sampled file is uncompressed (Explicit VR Little Endian), although the data page also lists JPEG Lossless, JPEG 2000 and Implicit VR files. Section 6.1 checks decoding.
- Image size varies a lot: 109 distinct sizes in the sample. 512x512 is the most common (35%), followed by 640x640 and 384x384, and the top ten run from 256x256 to 1024x1024. About 7% of sampled series (326) have non-square images, although the pixels themselves are square in every sampled series. In-plane pixel spacing has a median of 0.32 mm and runs from about 0.17 to 0.56 mm (5th to 95th percentile). The field of view mostly sits between 130 and 200 mm, with a clear peak at 160 mm.
- Slices are usually 3 mm thick with a small gap, typically 0.2 to 0.5 mm and sometimes about 1 mm. The thin axial slices (under 1 mm) come from 3D acquisitions.
- The TE/TR plot shows the familiar split: the short-TR, short-TE group (T1-like) is almost entirely flagged 0, and the long-TR group is mostly flagged 1. But 45% of the series flagged 0 have a long TR, meaning PD or T2-type contrast without fat suppression. In practice the flag behaves much like a fat-suppression flag, which fits with it matching `Fat_Suppression` exactly in train.
- `Anatomical_Plane` agrees with the orientation stored in the headers for every sampled series. Many acquisitions are tilted: the median tilt from the nearest scanner axis is about 7 degrees for sagittal, 8 for coronal and 4 for axial series, and a small group of coronal series is tilted by 30 to 40 degrees.
- 3D acquisitions are rare (139 of 4,429 sampled series) but explain the long tail of slice counts: a median of 144 slices against 30 for 2D.
- `SeriesDescription` is not dependable: 7.5% are blank, another 9.9% hold the placeholder `DummySeriesDesc!`, and the rest use hundreds of vendor-specific names (439 distinct values in the sample). The CSV columns are a cleaner source for plane and sequence type.
- `Laterality` is blank for about half the studies; the rest split evenly between left and right knees. `PatientSex` is roughly balanced. Each `PatientID` in the sample belongs to exactly one study (the IDs are anonymised, so this does not prove there are no repeat patients). All images are MONOCHROME2, stored with 12 or 16 bits, one frame per file.

## 6. Images

Now the pixels. One thing to know before plotting anything: the `.dcm` file names are random UIDs, so a sorted directory listing does not give the slices in anatomical order. The helper below reads every header in a series and sorts the slices by their position along the slice normal (`ImagePositionPatient` projected on the normal from `ImageOrientationPatient`), falling back to `InstanceNumber` when positions are missing.

Images are shown with a simple per-image window (0.5th to 99.5th intensity percentile), with the pixel aspect ratio taken from `PixelSpacing`.

In [ ]:
_ORDER_CACHE = {}


def ordered_files(split, study, series):
    """Slice paths of one series in anatomical order, with their positions (mm) and InstanceNumbers."""
    key = (split, study, series)
    if key in _ORDER_CACHE:
        return _ORDER_CACHE[key]
    sdir = series_dir(split, study, series)
    files = list_dcm(sdir)
    pos, inst, normal = [], [], None
    for f in files:
        try:
            d = pydicom.dcmread(os.path.join(sdir, f), stop_before_pixels=True)
        except Exception:
            pos.append(np.nan)
            inst.append(np.nan)
            continue
        inst.append(fnum(d.get('InstanceNumber', None)))
        try:
            if normal is None:
                o = np.asarray([float(v) for v in d.ImageOrientationPatient])
                normal = np.cross(o[:3], o[3:])
                if normal[np.argmax(np.abs(normal))] < 0:
                    normal = -normal
            pos.append(float(np.dot([float(v) for v in d.ImagePositionPatient], normal)))
        except Exception:
            pos.append(np.nan)
    pos, inst = np.asarray(pos, float), np.asarray(inst, float)
    if len(pos) and np.isfinite(pos).all():
        order = np.argsort(pos, kind='stable')
    elif len(inst) and np.isfinite(inst).all():
        order = np.argsort(inst, kind='stable')
    else:
        order = np.arange(len(files))
    out = {'files': [os.path.join(sdir, files[i]) for i in order], 'pos': pos[order], 'inst': inst[order],
           'file_rank': order}
    _ORDER_CACHE[key] = out
    return out


def load_slice(path):
    """Pixel array (float32), display aspect ratio and the dataset."""
    d = pydicom.dcmread(path)
    a = d.pixel_array
    if a.ndim == 3:
        a = a[..., 0] if a.shape[-1] in (3, 4) else a[a.shape[0] // 2]
    a = a.astype(np.float32)
    if str(d.get('PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
        a = a.max() - a
    ps_r, ps_c = two(d.get('PixelSpacing', None))
    aspect = ps_r / ps_c if (ps_r > 0 and ps_c > 0) else 1.0
    return a, aspect, d


def window(a, lo=0.5, hi=99.5):
    l, h = np.percentile(a, [lo, hi])
    return np.clip((a - l) / max(h - l, 1e-6), 0, 1)


def mid_slice(split, study, series):
    o = ordered_files(split, study, series)
    if not o['files']:
        raise FileNotFoundError('empty series')
    i = len(o['files']) // 2
    a, aspect, d = load_slice(o['files'][i])
    return a, aspect, d, i, len(o['files'])


_BY_STUDY = {k: g for k, g in S[ON_DISK].groupby('StudyInstanceUID')}


def candidates(study, plane, series_df=None, prefer_fs=1):
    """Series of one study and plane in display order: fluid-sensitive first, then by SeriesInstanceUID."""
    if series_df is None:
        c = _BY_STUDY.get(study, S.iloc[:0])
    else:
        c = series_df[(series_df.StudyInstanceUID == study) & (series_df.n_slices > 0)]
    c = c[c.Anatomical_Plane == plane]
    c = c.assign(pref=(c.Fluid_Sensitive == prefer_fs).astype(int))
    return c.sort_values(['pref', 'SeriesInstanceUID'], ascending=[False, True])


def pick_series(study, plane, series_df=None, prefer_fs=1):
    c = candidates(study, plane, series_df, prefer_fs)
    return None if c.empty else c.iloc[0]


DECODE_ERRORS = Counter()


def first_decodable(split, study, plane, series_df=None):
    """Middle slice of the first candidate series that decodes: (series row, pixels, aspect, slice index, n)."""
    for _, r in candidates(study, plane, series_df).iterrows():
        try:
            a, asp, d, k, n = mid_slice(split, study, r.SeriesInstanceUID)
            return r, a, asp, k, n
        except Exception as e:
            DECODE_ERRORS[type(e).__name__] += 1
    raise ValueError(f'no decodable {plane.lower()} series')


def blank(ax, text='not available'):
    ax.text(0.5, 0.5, text, ha='center', va='center', fontsize=9, color='#898781', transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


def show(ax, a, aspect, title):
    ax.imshow(window(a), cmap='gray', aspect=aspect, interpolation='antialiased')
    ax.set_title(title, fontsize=9, fontweight='normal')
    ax.axis('off')

### 6.1 Do the pixels decode?

The data page lists four pixel encodings (transfer syntaxes): uncompressed explicit and implicit VR, JPEG Lossless and JPEG 2000. The compressed ones need a decoder plugin for pydicom, so it is worth checking that one file of each kind decodes in this environment before relying on the pixels.

In [ ]:
def decode_check():
    libs = {}
    for m in ['pylibjpeg', 'libjpeg', 'openjpeg', 'gdcm', 'PIL']:
        try:
            __import__(m)
            libs[m] = 'yes'
        except Exception:
            libs[m] = 'no'
    print('pydicom', pydicom.__version__, '| decoder packages importable:', libs)
    rows = []
    for ts, g in hdr.groupby('transfer_syntax'):
        r = g.sort_values('SeriesInstanceUID').iloc[0]
        sdir = series_dir('train', r.StudyInstanceUID, r.SeriesInstanceUID)
        try:
            a, _, _ = load_slice(os.path.join(sdir, list_dcm(sdir)[0]))
            status = f'decoded, {a.shape[0]}x{a.shape[1]}'
        except Exception as e:
            status = f'failed: {type(e).__name__}: {str(e)[:80]}'
        rows.append({'transfer syntax': ts, 'series in sample': len(g), 'test decode of one file': status})
    display(pd.DataFrame(rows).set_index('transfer syntax'))


run('decode check', decode_check)

### 6.2 Does file order match anatomical order?

For a seeded sample of series, every header is read and two orderings are compared with the anatomical order given by `ImagePositionPatient`: the sorted file names and `InstanceNumber`. The numbers are Spearman rank correlations with the position order (1 = same order, -1 = reversed, around 0 = unrelated).

In [ ]:
def spearman(a, b):
    ra, rb = pd.Series(a).rank().values, pd.Series(b).rank().values
    if np.std(ra) == 0 or np.std(rb) == 0:
        return np.nan
    return float(np.corrcoef(ra, rb)[0, 1])


def check_order():
    rng_o = np.random.default_rng(SEED + 1)
    pool = hdr[['StudyInstanceUID', 'SeriesInstanceUID', 'plane']].reset_index(drop=True)
    idx = rng_o.choice(len(pool), size=min(N_ORDER_SERIES, len(pool)), replace=False)
    res = []
    for r in pool.iloc[np.sort(idx)].itertuples(index=False):
        o = ordered_files('train', r.StudyInstanceUID, r.SeriesInstanceUID)
        n = len(o['files'])
        if n < 3:
            continue
        rank = np.arange(n)
        fn_rho = spearman(o['file_rank'], rank)
        inst_rho = spearman(o['inst'], rank) if np.isfinite(o['inst']).all() else np.nan
        steps = np.diff(o['pos'])
        res.append({'plane': r.plane, 'slices': n, 'file name order': fn_rho, 'InstanceNumber order': inst_rho,
                    'duplicate positions': int(n - len(np.unique(np.round(o['pos'], 3)))),
                    'step mm (median)': float(np.median(steps)) if len(steps) else np.nan,
                    'step spread mm (max - min)': float(steps.max() - steps.min()) if len(steps) else np.nan})
    res = pd.DataFrame(res)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    bins = np.linspace(-1, 1, 41)
    ax.hist(res['file name order'].dropna(), bins=bins, color='#eda100', alpha=0.85, label='file name order')
    ax.hist(res['InstanceNumber order'].dropna(), bins=bins, color='#4a3aa7', alpha=0.85, label='InstanceNumber order')
    ax.set(title=f'Rank correlation with anatomical (position) order, {len(res)} series',
           xlabel='Spearman correlation', ylabel='series')
    ax.legend(frameon=False)
    finish(fig, '06_slice_order')
    print(f"|rho| > 0.99 for file names: {(res['file name order'].abs() > 0.99).mean():.0%}, "
          f"for InstanceNumber: {(res['InstanceNumber order'].abs() > 0.99).mean():.0%} "
          f"(InstanceNumber reversed in {(res['InstanceNumber order'] < -0.99).mean():.0%})")
    print(f"series with repeated slice positions: {(res['duplicate positions'] > 0).sum()} of {len(res)}; "
          f"series with uneven steps (> 0.1 mm spread): {(res['step spread mm (max - min)'] > 0.1).sum()}")
    return res


order_res = run('slice order', check_order)

### 6.3 The three planes side by side

Four studies picked at random (seeded) from those that have all three planes. For each plane the fluid-sensitive series is shown when the study has one, otherwise the first available series. Each image is the middle slice of the series in anatomical order.

In [ ]:
def plot_montage():
    rng_m = np.random.default_rng(SEED + 2)
    planes_on_disk = S[ON_DISK].groupby('StudyInstanceUID').Anatomical_Plane.apply(set)
    full = [st for st in sample_studies if set(PLANES) <= planes_on_disk.get(st, set())]
    chosen = sorted(rng_m.choice(full, size=min(4, len(full)), replace=False))
    fig, axes = plt.subplots(len(chosen), 3, figsize=(12, 4.1 * len(chosen)))
    axes = np.atleast_2d(axes)
    for i, st in enumerate(chosen):
        for j, p in enumerate(PLANES):
            ax = axes[i, j]
            try:
                s, a, asp, k, n = first_decodable('train', st, p)
                fs = 'FS' if s.Fluid_Sensitive == 1 else 'non-FS'
                show(ax, a, asp, f'study ...{st[-6:]} | {p} {fs}\nslice {k + 1}/{n}, {a.shape[0]}x{a.shape[1]}')
            except Exception as e:
                blank(ax, f'{p}: {type(e).__name__}')
    fig.suptitle('Middle slice of one series per plane (FS = fluid-sensitive)', fontweight='bold', y=1.0)
    finish(fig, '06_montage')
    return chosen


montage_studies = run('montage', plot_montage) or []

### 6.4 A strip through one series

Twelve evenly spaced slices through one sagittal series, in anatomical order, with each slice's position along the normal. This gives a feel for how much of the knee one series covers and how quickly the anatomy changes from slice to slice.

In [ ]:
def plot_strip():
    for cand in list(montage_studies) + list(sample_studies):
        try:
            s, _, _, _, n = first_decodable('train', cand, 'Sagittal')
        except Exception:
            continue
        if n >= 12:
            st = cand
            break
    else:
        raise ValueError('no sagittal series with at least 12 slices')
    o = ordered_files('train', st, s.SeriesInstanceUID)
    n = len(o['files'])
    idx = np.unique(np.linspace(0, n - 1, 12).round().astype(int))
    fig, axes = plt.subplots(2, 6, figsize=(17, 6.4))
    for ax, k in zip(axes.ravel(), idx):
        try:
            a, asp, _ = load_slice(o['files'][k])
            show(ax, a, asp, f'slice {k + 1}/{n}, {o["pos"][k]:.1f} mm')
        except Exception as e:
            blank(ax, type(e).__name__)
    for ax in axes.ravel()[len(idx):]:
        ax.axis('off')
    fs = 'fluid-sensitive' if s.Fluid_Sensitive == 1 else 'not fluid-sensitive'
    fig.suptitle(f'Sagittal {fs} series of study ...{st[-6:]}: 12 of {n} slices', fontweight='bold', y=1.0)
    finish(fig, '06_slice_strip')


run('slice strip', plot_strip)

### 6.5 Same view, different scanners

The same kind of image (a sagittal fluid-sensitive series, middle slice) from one study per scanner vendor in the header sample, plus one 3 T example. Field of view, resolution, contrast and noise all vary a lot.

In [ ]:
def plot_vendors():
    h = hdr[(hdr.plane == 'Sagittal') & (hdr.fs == 1)].sort_values(['StudyInstanceUID', 'SeriesInstanceUID'])
    vendors = [v for v in hdr['vendor'].value_counts().index if v not in ('Missing', 'Other')][:5]
    groups = [h[h.vendor == v] for v in vendors]
    used = set()
    picks = []
    for g in groups + [h[h.field == '3 T']]:
        for _, r in g.iterrows():
            if r.SeriesInstanceUID in used:
                continue
            try:
                a, asp, d, k, n = mid_slice('train', r.StudyInstanceUID, r.SeriesInstanceUID)
            except Exception as e:
                DECODE_ERRORS[type(e).__name__] += 1
                continue
            picks.append((r, a, asp))
            used.add(r.SeriesInstanceUID)
            break
        if len(picks) == 6:
            break
    fig, axes = plt.subplots(1, len(picks), figsize=(3.3 * len(picks), 3.9))
    for ax, (r, a, asp) in zip(np.atleast_1d(axes), picks):
        field = 'field strength n/a' if r.field == 'Missing' else r.field
        show(ax, a, asp, f'{r.vendor}, {field}\n{a.shape[0]}x{a.shape[1]}, {r.ps_row:.2f} mm pixels')
    fig.suptitle('Sagittal fluid-sensitive middle slices from different scanners', fontweight='bold', y=1.02)
    finish(fig, '06_vendors')


run('vendors', plot_vendors)

### 6.6 Intensity ranges

MRI intensities have no fixed unit, so the raw numbers depend on the scanner, the sequence and its settings. Below, for a seeded sample of series per plane and sequence type, the left plot shows the 99.5th percentile of the raw stored pixel values (before any rescale) of the middle slice. The right plot shows the distribution of pixel values after dividing each slice by that percentile, leaving out near-black background (below 5%).

In [ ]:
def plot_intensity():
    rng_i = np.random.default_rng(SEED + 3)
    rows, samples = [], {0: [], 1: []}
    for p in PLANES:
        for fs in [1, 0]:
            g = hdr[(hdr.plane == p) & (hdr.fs == fs)].reset_index(drop=True)
            if g.empty:
                continue
            for i in np.sort(rng_i.choice(len(g), size=min(N_INTENSITY_SERIES, len(g)), replace=False)):
                r = g.iloc[i]
                try:
                    a, _, d, _, _ = mid_slice('train', r.StudyInstanceUID, r.SeriesInstanceUID)
                except Exception as e:
                    DECODE_ERRORS[type(e).__name__] += 1
                    continue
                p995 = float(np.percentile(a, 99.5))
                rows.append({'plane': p, 'fs': fs, 'vendor': r.vendor, 'p99.5': p995, 'max': float(a.max())})
                if p995 > 0:
                    v = (a / p995).ravel()
                    v = v[v > 0.05]
                    if v.size:
                        samples[fs].append(v[rng_i.choice(v.size, size=min(4000, v.size), replace=False)])
    res = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.6), gridspec_kw={'width_ratios': [1.2, 1]})
    order = res['vendor'].value_counts().index.tolist()
    for fs in [1, 0]:
        v = res[res.fs == fs]
        xs = [order.index(x) + (0.15 if fs == 1 else -0.15) + rng_i.uniform(-0.08, 0.08) for x in v['vendor']]
        axes[0].scatter(xs, v['p99.5'], s=24, color=FS_COLORS[fs], alpha=0.85, edgecolor='white', linewidth=0.4,
                        label=FS_NAMES[fs])
    axes[0].set_xticks(range(len(order)))
    axes[0].set_xticklabels([f'{o}\n(n = {(res.vendor == o).sum()})' for o in order])
    axes[0].set_yscale('log')
    ticks = [t for t in [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]
             if res['p99.5'].min() / 2 <= t <= res['p99.5'].max() * 2]
    axes[0].set_yticks(ticks)
    axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    axes[0].yaxis.set_minor_formatter(plt.NullFormatter())
    axes[0].set(title='99.5th percentile of raw pixel values, middle slice', ylabel='raw value (log scale)')
    axes[0].legend(frameon=False)
    bins = np.linspace(0.05, 1.3, 64)
    for fs in [1, 0]:
        if samples[fs]:
            axes[1].hist(np.clip(np.concatenate(samples[fs]), 0, 1.3), bins=bins, density=True, histtype='step', lw=2,
                         color=FS_COLORS[fs], label=FS_NAMES[fs])
    axes[1].set(title="Pixel values / slice 99.5th percentile (background left out)", xlabel='relative intensity',
                ylabel='density')
    axes[1].legend(frameon=False)
    finish(fig, '06_intensity')
    print(f'{len(res)} slices; raw 99.5th percentile ranges from {res["p99.5"].min():,.0f} to {res["p99.5"].max():,.0f}')
    display(res.groupby('vendor')['p99.5'].describe()[['count', 'min', '50%', 'max']].round(0))


run('intensity', plot_intensity)

### 6.7 Labelled examples: positive vs negative

A few labelled studies for three findings, positives on the left and negatives on the right, drawn at random with a fixed seed (no cherry-picking). Each image is the **middle slice only** of a sagittal or coronal series (fluid-sensitive when available). A finding can sit well away from the middle slice, so this is a feel for the data, not a check of the labels.

In [ ]:
GALLERY = [('Effusion', 'Sagittal'), ('ACL', 'Sagittal'), ('Medial OA', 'Coronal')]
N_EACH = 3


def plot_gallery():
    rng_g = np.random.default_rng(SEED + 4)
    fig, axes = plt.subplots(len(GALLERY), 2 * N_EACH, figsize=(17, 3.3 * len(GALLERY)))
    for i, (label, plane) in enumerate(GALLERY):
        col = 0
        for val in [1, 0]:
            tag = 'positive' if val == 1 else 'negative'
            ids = sorted(lab.loc[lab[label] == val, 'StudyInstanceUID'])
            ids = [ids[j] for j in rng_g.permutation(len(ids))]
            shown = 0
            for sid in ids:
                if shown == N_EACH:
                    break
                try:
                    s, a, asp, k, n = first_decodable('train', sid, plane)
                except Exception:
                    continue
                ax = axes[i, col + shown]
                show(ax, a, asp, f'{label}: {tag}\n{plane}, slice {k + 1}/{n}')
                ax.axis('on')
                ax.grid(False)
                ax.set_xticks([])
                ax.set_yticks([])
                for sp in ax.spines.values():
                    sp.set_color(POS_COLOR if val == 1 else NEG_COLOR)
                    sp.set_linewidth(3 if val == 1 else 1.5)
                shown += 1
            for j in range(shown, N_EACH):
                blank(axes[i, col + j], f'no {tag} study with a {plane.lower()} series')
            col += N_EACH
    fig.suptitle('Labelled studies, middle slice only (thick red frame = positive, thin grey frame = negative)',
                 fontweight='bold', y=1.0)
    finish(fig, '06_labelled_examples')
    if DECODE_ERRORS:
        print('series skipped because they did not decode:', dict(DECODE_ERRORS))


run('labelled examples', plot_gallery)

**Observations**

- All sampled training files are uncompressed and decode with plain pydicom. The usual decoder plugins for JPEG Lossless and JPEG 2000 (pylibjpeg, gdcm) are not importable in this environment, so compressed files, if any turn up, would need one of them.
- File names carry no order at all (rank correlation around zero). `InstanceNumber` follows the anatomical order in every sampled series, but in 38% of them it counts in the opposite direction along the patient axis. The positions in the headers settle it either way. No sampled series had repeated positions or uneven slice spacing.
- One series covers the whole knee and some soft tissue on either side: in the strip above, 37 slices span about 120 mm.
- The montage and the scanner row show how much field of view, resolution, contrast and noise vary between studies, even for the same plane and sequence type.
- Raw intensity scales differ a lot: the 99.5th percentile of a middle slice ranges from about 250 to 8,350, and the Canon or Toshiba series in the sample sit around ten times higher than the Siemens ones (medians about 6,050 and 635). After dividing each slice by its own percentile, fluid-sensitive and non-fluid-sensitive images still have clearly different intensity distributions, as their contrast would suggest.
- On a single middle slice none of the three findings is obvious every time. Effusion has the best chance, since fluid is bright on these sequences, while the ACL and medial OA examples look much like the negatives. Most findings sit on a few slices of one or two series, not on the middle slice.

## 7. Test set notes and pitfalls

The public `test.csv` and `test_series.csv` hold only three example studies. The data page says they are replaced by the real test data (about 1,300 studies) when a submission is scored, and that the `Report` column is not available at that stage. So the only things to check here are the format and that the example studies look like training studies.

In [ ]:
def show_test():
    if test is None or test_series is None:
        print('test files not found')
        return
    print(f'test.csv: {len(test)} studies; test_series.csv: {len(test_series)} series')
    print('test_series.csv has the same columns as train_series.csv:',
          list(test_series.columns) == [c for c in train_series.columns if c != 'n_slices'])
    print('test studies that also appear in train.csv:', len(set(test.StudyInstanceUID) & set(train.StudyInstanceUID)))
    ts = test_series.copy()
    ts['n_slices'] = [len(list_dcm(series_dir('test', a, b))) for a, b in zip(ts.StudyInstanceUID, ts.SeriesInstanceUID)]

    def syntax(row):
        files = list_dcm(series_dir('test', row.StudyInstanceUID, row.SeriesInstanceUID))
        if not files:
            return 'no files'
        try:
            d = pydicom.dcmread(os.path.join(series_dir('test', row.StudyInstanceUID, row.SeriesInstanceUID), files[0]),
                                stop_before_pixels=True)
            return d.file_meta.TransferSyntaxUID.name
        except Exception as e:
            return type(e).__name__

    ts['transfer_syntax'] = [syntax(r) for r in ts.itertuples(index=False)]
    summary = ts.groupby('StudyInstanceUID').agg(
        series=('SeriesInstanceUID', 'size'),
        planes=('Anatomical_Plane', lambda x: ', '.join(f'{p} x{c}' for p, c in Counter(x).most_common())),
        fluid_sensitive=('Fluid_Sensitive', 'sum'),
        slice_files=('n_slices', 'sum'),
        transfer_syntax=('transfer_syntax', lambda x: ', '.join(sorted(set(x)))))
    summary.index = ['...' + i[-6:] for i in summary.index]
    display(summary)
    if sample_sub is not None:
        print('sample_submission columns are StudyInstanceUID + the 12 labels in order:',
              list(sample_sub.columns) == ['StudyInstanceUID'] + LABELS)
        display(sample_sub.head(3))
    return ts


test_series_n = run('test files', show_test)

In [ ]:
def plot_test_montage():
    ts = test_series_n
    if ts is None or (ts.n_slices > 0).sum() == 0:
        print('no test DICOMs found next to test_series.csv, skipping the test montage')
        return
    studies = sorted(ts.StudyInstanceUID.unique())
    fig, axes = plt.subplots(len(studies), 3, figsize=(12, 4.1 * len(studies)))
    axes = np.atleast_2d(axes)
    for i, st in enumerate(studies):
        for j, p in enumerate(PLANES):
            ax = axes[i, j]
            if pick_series(st, p, series_df=ts) is None:
                blank(ax, f'no {p.lower()} series')
                continue
            try:
                s, a, asp, k, n = first_decodable('test', st, p, series_df=ts)
                fs = 'FS' if s.Fluid_Sensitive == 1 else 'non-FS'
                show(ax, a, asp, f'test study ...{st[-6:]} | {p} {fs}\nslice {k + 1}/{n}, {a.shape[0]}x{a.shape[1]}')
            except Exception as e:
                blank(ax, f'{p}: {type(e).__name__}')
    fig.suptitle('The example test studies, middle slice per plane', fontweight='bold', y=1.0)
    finish(fig, '07_test_montage')


run('test montage', plot_test_montage)

The three example test studies look like typical training studies: five series each, all three planes, the same kinds of sequences, and uncompressed files. One of their axial series is a 160-slice acquisition with 640x1280 images, a good reminder that not every image is square.

**Things worth keeping in mind**

These are observations about the data rather than recommendations.

- **Only 58 labelled studies.** Every label statistic above rests on them, and the data page warns that prevalence may differ between train, public test and private test.
- **No reports at test time.** The test CSV only holds study ids. The series table has the same columns as for train.
- **Study and series size vary.** Studies have 3 to 14 series and series have 11 to 320 slices, with 3D series several times longer than 2D ones. Every training study has all three planes, but nothing on the data page promises the same for the hidden test studies.
- **File order is not slice order.** File names are random UIDs and the direction of `InstanceNumber` varies. The positions in the headers give the anatomical order.
- **Mixed geometry.** Over a hundred image sizes in the sample, pixel spacing from about 0.17 to 0.56 mm, tilted planes, and non-square images in about 7% of series.
- **Mixed intensity scales.** Raw values differ by an order of magnitude between vendors and sequences.
- **Pixel encodings.** The sampled training files and the example test files are all uncompressed, but the data page lists JPEG Lossless and JPEG 2000 as well, and the decoder plugins for those are not importable in this notebook's environment.
- **Metadata gaps.** `SeriesDescription` is blank or a placeholder in about 17% of sampled series, `Laterality` is blank in about half the studies and field strength in 6%.
- **Fluid_Sensitive and Fat_Suppression** are identical in train, but the data page says that is not guaranteed for every case.

## 8. Summary

- **Size.** 4,407 training studies with 24,371 series and about 819,000 slice files, and about 1,300 hidden test studies. Reports exist for the training studies only, in many languages.
- **Labels.** Only 58 studies have labels. Effusion is the most common finding (60%) and MCL the rarest (16%). No labelled study is completely normal. The findings fall into a loose degenerative group (OA, meniscus tears, Baker's cyst) and an acute injury group (ACL, contusion, fracture), with effusion and synovitis often together.
- **Series.** Every study has sagittal, coronal and axial series, usually five series in all. Most series are 2D with about 30 slices of 3 mm, and a small share are long 3D acquisitions.
- **Scanners.** Four main vendors, 40 scanner models, 1.5 T and 3 T. This shows up as large differences in image size, resolution, field of view, tilt and raw intensity.
- **Handling the pixels.** Slice order has to come from the header positions, and the sampled training files are all uncompressed even though the data page lists compressed encodings too.

Comments and corrections are welcome.

In [ ]:
print(f'total runtime: {time.time() - T0:.0f}s; figures saved to ./{FIG_DIR}/ ({len(os.listdir(FIG_DIR))} files)')